# Loan Approval Prediction — ML Classification

**Goal:** Predict whether a loan application will be approved using applicant financial and demographic features.

Models compared:
- Logistic Regression
- K-Nearest Neighbors (KNN)
- Gaussian Naive Bayes

**Note:** This notebook keeps the original learning approach but organizes the workflow more cleanly and avoids fitting preprocessing on the test set.

## 1. Import libraries
We use pandas/numpy for data handling, seaborn/matplotlib for visualization, and scikit-learn for preprocessing, models and evaluation.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

## 2. Load and inspect the dataset
First, check the shape, data types and missing values before preprocessing.

In [ ]:
data = pd.read_csv("loan_approval_data.csv")

print("Shape:", data.shape)
display(data.head())
data.info()

print("\nMissing values:")
display(data.isnull().sum())

## 3. Basic EDA
Look at the target distribution and a few important numerical relationships before training models.

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=data, x="Loan_Approved")
plt.title("Loan Approval Distribution")
plt.show()

numeric_view = data.select_dtypes(include="number")
plt.figure(figsize=(10, 7))
sns.heatmap(numeric_view.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

## 4. Separate features and target
`Applicant_ID` is an identifier, so it is removed rather than used as a predictive feature.

Important: the train/test split is done **before fitting imputers, encoders or scalers**. This prevents information from the test set leaking into training.

In [ ]:
# Remove the identifier because it should not influence the prediction.
data = data.drop(columns=["Applicant_ID"])

X = data.drop(columns=["Loan_Approved"])
y = data["Loan_Approved"].map({"No": 0, "Yes": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

## 5. Build the preprocessing pipeline
Numerical missing values are filled with the training-set mean. Categorical missing values use the most frequent category. Categorical features are one-hot encoded and numerical features are standardized.

Using a `ColumnTransformer` keeps the preprocessing reproducible and ensures that transformations are learned only from the training data.

In [ ]:
numeric_features = X_train.select_dtypes(include="number").columns
categorical_features = X_train.select_dtypes(include="object").columns

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", drop="first"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

## 6. Train and evaluate Logistic Regression

In [ ]:
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

logistic_model.fit(X_train, y_train)
y_pred_logistic = logistic_model.predict(X_test)

print(classification_report(y_test, y_pred_logistic))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_logistic))

## 7. Train and evaluate KNN
KNN is sensitive to feature scale, which is why scaling is included in the preprocessing pipeline.

In [ ]:
knn_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", KNeighborsClassifier(n_neighbors=21))
])

knn_model.fit(X_train, y_train)
y_pred_knn = knn_model.predict(X_test)

print(classification_report(y_test, y_pred_knn))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_knn))

## 8. Train and evaluate Gaussian Naive Bayes
Gaussian Naive Bayes assumes that numerical features follow a Gaussian distribution.

In [ ]:
# GaussianNB needs a dense feature matrix, so the preprocessor is converted to dense output.
preprocessor_dense = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False), categorical_features)
])

nb_model = Pipeline([
    ("preprocessor", preprocessor_dense),
    ("model", GaussianNB())
])

nb_model.fit(X_train, y_train)
y_pred_nb = nb_model.predict(X_test)

print(classification_report(y_test, y_pred_nb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_nb))

## 9. Compare the models
Accuracy alone can be misleading, so compare precision, recall and F1-score as well.

In [ ]:
predictions = {
    "Logistic Regression": y_pred_logistic,
    "KNN": y_pred_knn,
    "Gaussian Naive Bayes": y_pred_nb
}

results = []
for name, predictions_for_model in predictions.items():
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions_for_model),
        "Precision": precision_score(y_test, predictions_for_model, zero_division=0),
        "Recall": recall_score(y_test, predictions_for_model, zero_division=0),
        "F1 Score": f1_score(y_test, predictions_for_model, zero_division=0)
    })

results_df = pd.DataFrame(results).sort_values("F1 Score", ascending=False)
display(results_df)

## 10. What I learned
- Different classification algorithms can perform differently on the same dataset.
- Preprocessing is an important part of an ML workflow.
- KNN is affected by feature scaling.
- Precision, recall and F1-score provide more information than accuracy alone.
- Pipelines help keep preprocessing consistent between training and testing.

### Next improvements
- Try cross-validation instead of relying on a single train/test split.
- Tune hyperparameters for KNN and Logistic Regression.
- Experiment with additional classification algorithms.
- Investigate feature importance/model interpretability.
- Deploy the best model using an API after the ML workflow is stable.